In [ ]:
import cv2
import matplotlib.pyplot as plt
import time
from datetime import datetime
from pathlib import Path

Path("Picture").mkdir(exist_ok=True)
cap = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION) #打开摄像头

# 3秒后开始拍照
start = time.time()
while time.time() - start < 3:
    ret, frame = cap.read() # 读取一帧图像
cap.release() # 释放摄像头

if not ret: raise RuntimeError("无法读取图像")

filename = f"Picture/Photo_{datetime.now():%Y%m%d_%H%M%S}.png"
cv2.imwrite(filename, frame)

plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

动态监测

In [ ]:
import cv2
import time
import threading
import ipywidgets as widgets
from ultralytics import YOLO
from datetime import datetime
from pathlib import Path
from IPython.display import display

Path("Picture").mkdir(exist_ok=True)
model = YOLO("yolo26n.pt")
stop_event = threading.Event()

preview = widgets.Image(format="jpeg", width=640)
stop_button = widgets.Button(description="Stop Camera", button_style="danger")
status = widgets.Label(value="摄像头正在启动...")

def stop_camera(button):
    stop_event.set()
    stop_button.disabled = True
    status.value = "正在关闭摄像头..."

stop_button.on_click(stop_camera)

def camera_loop():
    cap = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
    if not cap.isOpened():
        status.value = "无法打开摄像头"
        return

    armed = True
    last_seen = time.monotonic()

    try:
        while not stop_event.is_set():
            ret, frame = cap.read()
            if not ret: break

            height = int(frame.shape[0] * 640 / frame.shape[1])
            small = cv2.resize(frame, (640, height))
            result = model.predict(small, classes=[0], conf=0.5, imgsz=320, verbose=False)[0]

            now = time.monotonic()
            person_detected = len(result.boxes) > 0

            if person_detected:
                last_seen = now
                if armed:
                    filename = f"Picture/Photo_{datetime.now():%Y%m%d_%H%M%S}.png"
                    if cv2.imwrite(filename, frame):
                        status.value = f"检测到人员，已拍照：{filename}"
                        armed = False

            if not person_detected and now - last_seen >= 2:
                armed = True
                status.value = "等待人员进入画面..."

            success, buffer = cv2.imencode(".jpg", result.plot())
            if success: preview.value = buffer.tobytes()
            time.sleep(0.03)

    except Exception as e:
        status.value = f"运行错误：{e}"
    finally:
        cap.release()
        stop_button.disabled = True
        status.value = "摄像头已关闭"

display(widgets.VBox([preview, stop_button, status]))
camera_thread = threading.Thread(target=camera_loop, daemon=True)
camera_thread.start()

In [ ]:
import os
import ssl
import smtplib
import subprocess
import threading
import time
import ipywidgets as widgets
from pathlib import Path
from email.message import EmailMessage
from IPython.display import display

SENDER = "2310539989@qq.com"
RECEIVER = "yemingxin@sinap.ac.cn"
PASSWORD = subprocess.check_output(["security", "find-generic-password", "-a", SENDER, "-s", "codex-send-email-qq-smtp", "-w"], text=True).strip()

def SendPhoto(filename):
    msg = EmailMessage()
    msg["Subject"] = "Camera - Person Detected"
    msg["From"] = SENDER
    msg["To"] = RECEIVER
    msg.set_content("摄像头检测到人员，照片见附件。")
    path = Path(filename)
    msg.add_attachment(path.read_bytes(), maintype="image", subtype="png", filename=path.name)

    with smtplib.SMTP_SSL("smtp.qq.com", 465, context=ssl.create_default_context(), timeout=20) as smtp:
        smtp.login(SENDER, PASSWORD)
        smtp.send_message(msg)

mail_status = widgets.Label(value="邮件监控已启动，等待新照片...")
display(mail_status)

sent_photos = set(Path("Picture").glob("Photo_*.png"))
last_attempt = {}

def WatchPhotos():
    while not stop_event.is_set():
        for photo in sorted(Path("Picture").glob("Photo_*.png")):
            if photo in sent_photos: continue
            if time.time() - photo.stat().st_mtime < 1: continue
            if time.monotonic() - last_attempt.get(photo, 0) < 30: continue

            last_attempt[photo] = time.monotonic()
            try:
                SendPhoto(photo)
                sent_photos.add(photo)
                mail_status.value = f"邮件发送成功：{photo.name}"
            except Exception as e:
                mail_status.value = f"邮件发送失败：{e}"

        stop_event.wait(1)

    mail_status.value = "邮件监控已停止"

mail_thread = threading.Thread(target=WatchPhotos, daemon=True)
mail_thread.start()

连续监测

In [1]:
import cv2
import time
import ssl
import smtplib
import threading
import subprocess
import ipywidgets as widgets
from queue import Queue
from ultralytics import YOLO
from datetime import datetime
from pathlib import Path
from email.message import EmailMessage
from IPython.display import display

# 基本配置
SENDER = "2310539989@qq.com"
RECEIVER = "yemingxin@sinap.ac.cn"
PASSWORD = subprocess.check_output(["security", "find-generic-password", "-a", SENDER, "-s", "codex-send-email-qq-smtp", "-w"], text=True).strip()

PHOTO_INTERVAL = 1.0
MAX_PHOTOS_PER_GROUP = 5
EXIT_DELAY = 2.0
EDGE_RATIO = 0.15
MAX_EMAIL_BYTES = 10 * 1024 * 1024

Path("Picture").mkdir(exist_ok=True)
model = YOLO("yolo26n.pt")
stop_event = threading.Event()
mail_queue = Queue()

preview = widgets.Image(format="jpeg", width=640)
stop_button = widgets.Button(description="Stop Camera", button_style="danger")
status = widgets.Label(value="摄像头正在启动...")
mail_status = widgets.Label(value="邮件发送系统已启动")

def stop_camera(button):
    stop_event.set()
    stop_button.disabled = True
    status.value = "正在关闭摄像头..."

stop_button.on_click(stop_camera)

# 发送一组照片，附件过大时自动分成多封邮件
def SendGroup(folder, photos):
    batches = []
    batch = []
    size = 0

    for photo in photos:
        photo_size = photo.stat().st_size
        if batch and size + photo_size > MAX_EMAIL_BYTES:
            batches.append(batch)
            batch = []
            size = 0
        batch.append(photo)
        size += photo_size
    if batch: batches.append(batch)

    failures = 0
    for i, files in enumerate(batches, 1):
        try:
            msg = EmailMessage()
            msg["Subject"] = f"Camera Alert - {folder.name} ({i}/{len(batches)})"
            msg["From"] = SENDER
            msg["To"] = RECEIVER
            msg.set_content(f"检测到人员经过摄像头。\n事件：{folder.name}\n本组共拍摄 {len(photos)} 张照片。\n本邮件包含 {len(files)} 张照片。")

            for photo in files:
                msg.add_attachment(photo.read_bytes(), maintype="image", subtype="png", filename=photo.name)

            with smtplib.SMTP_SSL("smtp.qq.com", 465, context=ssl.create_default_context(), timeout=30) as smtp:
                smtp.login(SENDER, PASSWORD)
                smtp.send_message(msg)

            mail_status.value = f"邮件发送成功：{folder.name} ({i}/{len(batches)})"
        except Exception as e:
            failures += 1
            mail_status.value = f"邮件发送失败：{e}"

    if failures:
        mail_status.value = f"{folder.name} 有 {failures} 封邮件发送失败，照片已保存在本地"
    else:
        mail_status.value = f"{folder.name} 已发送，共 {len(photos)} 张照片"

# 邮件在独立线程中发送，不阻塞视频
def mail_loop():
    while True:
        job = mail_queue.get()
        if job is None:
            mail_queue.task_done()
            break
        folder, photos = job
        try:
            SendGroup(folder, photos)
        except Exception as e:
            mail_status.value = f"邮件任务失败：{e}"
        finally:
            mail_queue.task_done()

# 摄像头主循环
def camera_loop():
    cap = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
    state = "waiting"
    photos = []
    folder = None
    last_seen = time.monotonic()
    last_photo = 0

    try:
        if not cap.isOpened(): raise RuntimeError("无法打开摄像头")

        while not stop_event.is_set():
            ret, frame = cap.read()
            if not ret: break

            height = int(frame.shape[0] * 640 / frame.shape[1])
            small = cv2.resize(frame, (640, height))
            result = model.predict(small, classes=[0], conf=0.5, imgsz=320, verbose=False)[0]

            now = time.monotonic()
            people = len(result.boxes) > 0
            edge = False
            margin = int(640 * EDGE_RATIO)

            for box in result.boxes.xyxy.cpu().numpy():
                if box[0] <= margin or box[2] >= 640 - margin:
                    edge = True
                    break

            if people: last_seen = now

            # 等待人物从左右边缘进入
            if state == "waiting" and people:
                if edge:
                    state = "capturing"
                    folder = Path("Picture") / f"Group_{datetime.now():%Y%m%d_%H%M%S}"
                    folder.mkdir(exist_ok=True)
                    photos = []
                    last_photo = now - PHOTO_INTERVAL
                    status.value = f"检测到人员进入，开始拍照：{folder.name}"
                else:
                    state = "ignore"

            # 已经在画面中央的人不触发新的进入事件
            if state == "ignore" and not people and now - last_seen >= EXIT_DELAY:
                state = "waiting"
                status.value = "等待下一位人员从边缘进入..."

            # 开始拍摄后，人物在画面任何位置都可以继续拍照
            if state == "capturing":

                # 每组最多拍摄 MAX_PHOTOS_PER_GROUP 张
                if people and len(photos) < MAX_PHOTOS_PER_GROUP and now - last_photo >= PHOTO_INTERVAL:
                    filename = folder / f"Photo_{datetime.now():%H%M%S_%f}.png"
                    if cv2.imwrite(str(filename), frame):
                        photos.append(filename)
                        last_photo = now
                        status.value = f"正在拍照：第 {len(photos)}/{MAX_PHOTOS_PER_GROUP} 张"

                        if len(photos) == MAX_PHOTOS_PER_GROUP:
                            status.value = f"已达到每组 {MAX_PHOTOS_PER_GROUP} 张上限，等待人员离开..."

                # 连续 2 秒没有检测到人，结束当前事件
                if not people and now - last_seen >= EXIT_DELAY:
                    if photos: mail_queue.put((folder, photos.copy()))
                    status.value = f"人员已离开，本组共 {len(photos)} 张照片"
                    state = "waiting"
                    photos = []

            # 显示 YOLO 检测框和边缘触发线
            display_frame = result.plot()
            cv2.line(display_frame, (margin, 0), (margin, height), (0, 165, 255), 2)
            cv2.line(display_frame, (640 - margin, 0), (640 - margin, height), (0, 165, 255), 2)

            success, buffer = cv2.imencode(".jpg", display_frame)
            if success: preview.value = buffer.tobytes()
            time.sleep(0.03)

    except Exception as e:
        status.value = f"运行错误：{e}"
    finally:
        if state == "capturing" and photos:
            mail_queue.put((folder, photos.copy()))
        cap.release()
        mail_queue.put(None)
        stop_button.disabled = True
        status.value = "摄像头已关闭"

display(widgets.VBox([preview, stop_button, status, mail_status]))

mail_thread = threading.Thread(target=mail_loop, daemon=True)
camera_thread = threading.Thread(target=camera_loop, daemon=True)
mail_thread.start()
camera_thread.start()